In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

# paths
OHIO_PATH = os.path.expanduser('~/Desktop/diabloom/data/raw/ohiot1dm')
MODEL_SAVE_PATH = os.path.expanduser('~/Desktop/diabloom/models/glucose_lstm_ohio_final.pt')

# constants
GLUCOSE_MIN, GLUCOSE_MAX = 40, 400
HYPO_THRESHOLD = 70
WINDOW_SIZE = 24
HORIZON = 6
IOB_MAX = 20.0

FEATURE_COLS = ['glucose_norm', 'delta_1', 'delta_3', 'delta_6',
                'rolling_mean_12', 'rolling_std_12',
                'hour_sin', 'hour_cos', 'iob_norm']

PATIENT_IDS = ['559', '563', '570', '575', '588', '591']

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Patients: {PATIENT_IDS}")
print(f"Features: {len(FEATURE_COLS)}")

Device: mps
Patients: ['559', '563', '570', '575', '588', '591']
Features: 9


In [4]:
def parse_ohio_glucose(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    records = []
    for entry in root.find('glucose_level'):
        records.append({
            'timestamp': pd.to_datetime(
                entry.attrib['ts'], format='%d-%m-%Y %H:%M:%S'
            ),
            'glucose_mgdl': float(entry.attrib['value'])
        })
    df = pd.DataFrame(records).sort_values('timestamp').reset_index(drop=True)
    return df

def parse_ohio_bolus(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    records = []
    for entry in root.find('bolus'):
        records.append({
            'timestamp': pd.to_datetime(
                entry.attrib['ts_begin'], format='%d-%m-%Y %H:%M:%S'
            ),
            'dose': float(entry.attrib['dose'])
        })
    df = pd.DataFrame(records).sort_values('timestamp').reset_index(drop=True)
    return df

def compute_iob(glucose_df, bolus_df, 
                peak_mins=60, duration_mins=240):
    boluses = bolus_df[bolus_df['dose'] > 0].copy()
    glucose_df = glucose_df.copy()
    iob_values = []
    for t in glucose_df['timestamp']:
        total_iob = 0.0
        for _, bolus in boluses.iterrows():
            mins_elapsed = (t - bolus['timestamp']).total_seconds() / 60
            if 0 <= mins_elapsed <= duration_mins:
                remaining = max(0, 1 - (mins_elapsed / duration_mins))
                total_iob += bolus['dose'] * remaining
        iob_values.append(total_iob)
    glucose_df['iob'] = iob_values
    return glucose_df

# load all patients
print("Loading patients...")
ohio_data = {}
for fname in sorted(os.listdir(OHIO_PATH)):
    if 'training' in fname and fname.endswith('.xml'):
        pid = fname.split('-')[0]
        fpath = os.path.join(OHIO_PATH, fname)
        glucose = parse_ohio_glucose(fpath)
        bolus = parse_ohio_bolus(fpath)
        ohio_data[pid] = {'glucose': glucose, 'bolus': bolus}
        print(f"Patient {pid}: {len(glucose)} glucose readings, "
              f"{len(bolus)} bolus records")

print(f"\nLoaded {len(ohio_data)} patients")

Loading patients...
Patient 559: 10796 glucose readings, 152 bolus records
Patient 563: 12124 glucose readings, 347 bolus records
Patient 570: 10982 glucose readings, 326 bolus records
Patient 575: 11866 glucose readings, 187 bolus records
Patient 588: 12640 glucose readings, 182 bolus records
Patient 591: 10847 glucose readings, 261 bolus records

Loaded 6 patients


In [5]:
# compute IOB for all patients
print("Computing IOB (this will take a few minutes)...")
ohio_iob_data = {}

for pid, data in ohio_data.items():
    glucose_df = data['glucose'].copy()
    bolus_df = data['bolus'].copy()
    
    glucose_with_iob = compute_iob(glucose_df, bolus_df)
    ohio_iob_data[pid] = glucose_with_iob
    
    nonzero = (glucose_with_iob['iob'] > 0).sum()
    pct = 100 * nonzero / len(glucose_with_iob)
    print(f"Patient {pid}: IOB non-zero: {nonzero}/{len(glucose_with_iob)} "
          f"({pct:.1f}%) | max IOB: {glucose_with_iob['iob'].max():.2f}U")

print("\nIOB computation complete")

Computing IOB (this will take a few minutes)...
Patient 559: IOB non-zero: 5727/10796 (53.0%) | max IOB: 12.59U
Patient 563: IOB non-zero: 8998/12124 (74.2%) | max IOB: 48.22U
Patient 570: IOB non-zero: 7110/10982 (64.7%) | max IOB: 26.69U
Patient 575: IOB non-zero: 6975/11866 (58.8%) | max IOB: 16.40U
Patient 588: IOB non-zero: 7333/12640 (58.0%) | max IOB: 9.93U
Patient 591: IOB non-zero: 7745/10847 (71.4%) | max IOB: 10.34U

IOB computation complete


In [6]:
# D1NAMO normalisation stats (computed earlier, hardcoded here)
# these ensure OhioT1DM features are on the same scale as D1NAMO training
NORM_STATS = {
    'delta_1':        {'mean': 0.0281,  'std': 5.6046},
    'delta_3':        {'mean': 0.0985,  'std': 15.2474},
    'delta_6':        {'mean': 0.2321,  'std': 28.2503},
    'rolling_mean_12':{'mean': 166.1347,'std': 74.5364},
    'rolling_std_12': {'mean': 12.0754, 'std': 11.2882}
}

def engineer_features(glucose_iob_df):
    df = glucose_iob_df.copy().sort_values('timestamp').reset_index(drop=True)
    
    df['glucose_norm'] = (df['glucose_mgdl'] - GLUCOSE_MIN) / (GLUCOSE_MAX - GLUCOSE_MIN)
    df['delta_1'] = df['glucose_mgdl'].diff(1)
    df['delta_3'] = df['glucose_mgdl'].diff(3)
    df['delta_6'] = df['glucose_mgdl'].diff(6)
    df['rolling_mean_12'] = df['glucose_mgdl'].rolling(12).mean()
    df['rolling_std_12'] = df['glucose_mgdl'].rolling(12).std()
    df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
    df['iob_norm'] = np.clip(df['iob'] / IOB_MAX, 0, 1)
    
    # normalise delta and rolling features using D1NAMO stats
    for col in ['delta_1', 'delta_3', 'delta_6',
                'rolling_mean_12', 'rolling_std_12']:
        mean = NORM_STATS[col]['mean']
        std = NORM_STATS[col]['std']
        df[col] = (df[col] - mean) / (std + 1e-8)
    
    df = df.dropna().reset_index(drop=True)
    return df

def create_windows(df, feature_cols, 
                   window_size=WINDOW_SIZE, horizon=HORIZON):
    data = df[feature_cols].values
    target = df['glucose_norm'].values
    X, y = [], []
    for i in range(len(data) - window_size - horizon):
        X.append(data[i:i + window_size])
        y.append(target[i + window_size + horizon])
    return np.array(X), np.array(y)

# build windows with 80/20 temporal split
X_train_all, y_train_all = [], []
X_val_all, y_val_all = [], []

for pid, glucose_iob_df in ohio_iob_data.items():
    features_df = engineer_features(glucose_iob_df)
    
    split = int(len(features_df) * 0.8)
    train_df = features_df.iloc[:split]
    val_df = features_df.iloc[split:]
    
    X_tr, y_tr = create_windows(train_df, FEATURE_COLS)
    X_val, y_val = create_windows(val_df, FEATURE_COLS)
    
    if X_tr.shape[0] > 0:
        X_train_all.append(X_tr)
        y_train_all.append(y_tr)
    if X_val.shape[0] > 0:
        X_val_all.append(X_val)
        y_val_all.append(y_val)
    
    print(f"Patient {pid}: train {X_tr.shape[0]}, val {X_val.shape[0]}")

X_train = np.concatenate(X_train_all)
y_train = np.concatenate(y_train_all)
X_val = np.concatenate(X_val_all)
y_val = np.concatenate(y_val_all)

print(f"\nFinal training set: {X_train.shape}")
print(f"Final validation set: {X_val.shape}")
print(f"Feature dimension: {X_train.shape[2]} (should be 9)")

Patient 559: train 8598, val 2127
Patient 563: train 9660, val 2393
Patient 570: train 8746, val 2165
Patient 575: train 9454, val 2341
Patient 588: train 10073, val 2496
Patient 591: train 8638, val 2138

Final training set: (55169, 24, 9)
Final validation set: (13660, 24, 9)
Feature dimension: 9 (should be 9)


In [7]:
class GlucoseLSTM(nn.Module):
    def __init__(self, input_size=9, hidden_size=64, 
                 num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze()

# tensors
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train)
X_v = torch.FloatTensor(X_val)
y_v = torch.FloatTensor(y_val)

train_loader = DataLoader(TensorDataset(X_tr, y_tr),
                         batch_size=32, shuffle=True)

# training setup
model = GlucoseLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on: {device}")
print(f"Training samples: {len(X_tr)}")
print("Starting training...\n")

best_val_loss = float('inf')
best_epoch = 0
train_history = []
val_history = []

for epoch in range(50):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    model.eval()
    with torch.no_grad():
        val_pred = model(X_v.to(device))
        val_loss = criterion(val_pred, y_v.to(device)).item()
    
    train_loss = np.mean(train_losses)
    train_history.append(train_loss)
    val_history.append(val_loss)
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train: {train_loss:.4f} "
              f"| val: {val_loss:.4f}")

print(f"\nBest val loss: {best_val_loss:.4f} at epoch {best_epoch}")
print(f"Model saved to: {MODEL_SAVE_PATH}")

Model parameters: 54,593
Training on: mps
Training samples: 55169
Starting training...



/Users/tavle/Desktop/diabloom/venv/lib/python3.14/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch   0 | train: 0.0070 | val: 0.0203
Epoch  10 | train: 0.0037 | val: 0.0072
Epoch  20 | train: 0.0024 | val: 0.0075
Epoch  30 | train: 0.0019 | val: 0.0082
Epoch  40 | train: 0.0018 | val: 0.0082

Best val loss: 0.0069 at epoch 9
Model saved to: /Users/tavle/Desktop/diabloom/models/glucose_lstm_ohio_final.pt
